# Inferência — máscara de instâncias e contagem de núcleos

Recebe o caminho de uma imagem qualquer (PNG/JPG/TIF, cinza ou colorida, qualquer tamanho) e devolve a máscara de instâncias colorida e a contagem. Usa o checkpoint `checkpoints/unet_3classes_bbbc038.pt` (U-Net de 3 classes + watershed, Parte 2 do `assigment1DL.ipynb`); não retreina nada.

Imagens maiores que 128×128 são processadas em tiles sobrepostos com fusão das probabilidades antes do watershed (Parte 4); menores são preenchidas por reflexão.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import nucleos as nu

CAMINHO_IMAGEM = "data/BBBC038/stage1_train/00071198d059ba7f5914a526d124d28e6d010c92466da21d4a04cd5413362552/images/00071198d059ba7f5914a526d124d28e6d010c92466da21d4a04cd5413362552.png"
CHECKPOINT = "checkpoints/unet_3classes_bbbc038.pt"

In [ ]:
modelo = nu.carregar_modelo(CHECKPOINT)
imagem, inst_mask = nu.segmentar_arquivo(modelo, CAMINHO_IMAGEM)
contagem = nu.n_objetos(inst_mask)
print(CAMINHO_IMAGEM)
print(f"{imagem.shape[1]}×{imagem.shape[0]} px | {contagem} núcleos")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(Image.open(CAMINHO_IMAGEM))
ax[0].set_title("imagem original")
ax[1].imshow(imagem, cmap="gray", vmin=0, vmax=1)
ax[1].set_title("entrada da rede (cinza normalizado)")
ax[2].imshow(nu.colorir_instancias(inst_mask))
ax[2].set_title(f"instâncias: {contagem} núcleos")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

Para salvar a máscara colorida (PNG) e o mapa de instâncias (id por pixel, `.npy`) ao lado da imagem de entrada:

In [ ]:
from pathlib import Path

saida = Path(CAMINHO_IMAGEM).with_suffix("")
Image.fromarray((nu.colorir_instancias(inst_mask) * 255).astype(np.uint8)).save(f"{saida}_instancias.png")
np.save(f"{saida}_instancias.npy", inst_mask)
print("salvo:", f"{saida}_instancias.png", "|", f"{saida}_instancias.npy")